# Descriptive Statistics

**Course:** Data Science  
**Lesson:** 04  
**Difficulty:** Beginner–Intermediate

## Learning Objectives

By the end of this notebook, students should be able to:

- distinguish descriptive from inferential statistics;
- calculate and interpret the mean, median, and mode;
- calculate range, variance, and standard deviation;
- interpret percentiles, quartiles, and the interquartile range;
- produce a five-number summary;
- detect potential outliers using the IQR rule;
- calculate and interpret skewness;
- choose suitable summary measures for different distributions.

## Required Libraries

- Pandas
- NumPy
- Matplotlib

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Load the Dataset

The dataset contains anonymized employee records. It is used only for teaching and was generated synthetically.

In [ ]:
candidate_paths = [
    Path("../datasets/employee_statistics.csv"),
    Path("Data-Science-Course/datasets/employee_statistics.csv"),
    Path("datasets/employee_statistics.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "employee_statistics.csv was not found. "
        "Place it in Data-Science-Course/datasets/."
    )

df = pd.read_csv(data_path, parse_dates=["Hire_Date"])
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

The dataset contains categorical, numerical, and date variables. This lesson focuses mainly on numerical variables.

## 2. Descriptive and Inferential Statistics

**Descriptive statistics** summarize the observed dataset.

Examples include:

- averages;
- percentages;
- measures of spread;
- tables and charts.

**Inferential statistics** use sample data to make conclusions about a larger population. Inferential methods are introduced later in the course.

## 3. Selecting Numerical Variables

In [ ]:
numerical_columns = [
    "Age",
    "Years_Experience",
    "Annual_Salary_USD",
    "Weekly_Hours",
    "Performance_Score",
    "Job_Satisfaction",
    "Projects_Completed",
    "Training_Hours",
    "Absence_Days",
]

df[numerical_columns].head()

## 4. Mean

The mean is the sum of all observations divided by the number of observations.

It is useful for approximately symmetric data but can be affected strongly by extreme values.

In [ ]:
mean_salary = df["Annual_Salary_USD"].mean()
mean_salary

The result represents the arithmetic average annual salary in the dataset.

## 5. Median

The median is the middle value after sorting the observations.

It is less sensitive to extreme values than the mean.

In [ ]:
median_salary = df["Annual_Salary_USD"].median()
median_salary

In [ ]:
pd.DataFrame({
    "Measure": ["Mean", "Median"],
    "Annual_Salary_USD": [mean_salary, median_salary]
})

If the mean is noticeably higher than the median, high salary values may be pulling the average upward.

## 6. Mode

The mode is the most frequently occurring value. A dataset may have one mode, several modes, or no unique mode.

In [ ]:
department_mode = df["Department"].mode()
department_mode

In [ ]:
age_modes = df["Age"].mode()
age_modes

The mode is especially useful for categorical variables, such as the most common department.

## 7. Comparing Mean, Median, and Mode

In [ ]:
salary_summary = pd.Series({
    "Mean": df["Annual_Salary_USD"].mean(),
    "Median": df["Annual_Salary_USD"].median(),
    "Mode": df["Annual_Salary_USD"].mode().iloc[0],
})

salary_summary

Use:

- the **mean** for balanced numerical distributions without influential extremes;
- the **median** for skewed data or data with outliers;
- the **mode** for the most common category or repeated numerical value.

## 8. Range

The range is the difference between the maximum and minimum values.

In [ ]:
salary_min = df["Annual_Salary_USD"].min()
salary_max = df["Annual_Salary_USD"].max()
salary_range = salary_max - salary_min

salary_min, salary_max, salary_range

The range is simple, but it depends only on two observations and is highly sensitive to extreme values.

## 9. Variance

Variance measures the average squared distance from the mean.

Pandas uses the sample variance by default, with `ddof=1`.

In [ ]:
salary_variance = df["Annual_Salary_USD"].var()
salary_variance

Variance is expressed in squared units, which makes direct interpretation less intuitive.

## 10. Standard Deviation

Standard deviation is the square root of variance. It uses the same units as the original variable.

In [ ]:
salary_std = df["Annual_Salary_USD"].std()
salary_std

In [ ]:
np.sqrt(salary_variance)

A larger standard deviation indicates that observations are more widely dispersed around the mean.

## 11. Comparing Variation Across Departments

In [ ]:
department_salary_stats = (
    df.groupby("Department")["Annual_Salary_USD"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
    .sort_values("mean", ascending=False)
)

department_salary_stats

The table shows both typical salary levels and the amount of variation within each department.

## 12. Percentiles

A percentile indicates the value below which a specified percentage of observations falls.

In [ ]:
salary_percentiles = df["Annual_Salary_USD"].quantile([0.10, 0.25, 0.50, 0.75, 0.90])
salary_percentiles

For example, the 90th percentile is the salary at or below which approximately 90% of observations fall.

## 13. Quartiles

Quartiles divide ordered data into four parts:

- Q1: 25th percentile;
- Q2: 50th percentile, or median;
- Q3: 75th percentile.

In [ ]:
q1 = df["Annual_Salary_USD"].quantile(0.25)
q2 = df["Annual_Salary_USD"].quantile(0.50)
q3 = df["Annual_Salary_USD"].quantile(0.75)

q1, q2, q3

## 14. Interquartile Range

The interquartile range describes the spread of the middle 50% of observations.

`IQR = Q3 - Q1`

In [ ]:
salary_iqr = q3 - q1
salary_iqr

The IQR is resistant to extreme values and is useful for skewed distributions.

## 15. Five-Number Summary

The five-number summary contains:

- minimum;
- Q1;
- median;
- Q3;
- maximum.

In [ ]:
five_number_summary = pd.Series({
    "Minimum": df["Annual_Salary_USD"].min(),
    "Q1": q1,
    "Median": q2,
    "Q3": q3,
    "Maximum": df["Annual_Salary_USD"].max(),
})

five_number_summary

## 16. Detecting Potential Outliers with the IQR Rule

The common IQR rule defines:

- lower bound = Q1 − 1.5 × IQR;
- upper bound = Q3 + 1.5 × IQR.

Values outside these bounds are potential outliers. They are not automatically errors.

In [ ]:
lower_bound = q1 - 1.5 * salary_iqr
upper_bound = q3 + 1.5 * salary_iqr

lower_bound, upper_bound

In [ ]:
salary_outliers = df.loc[
    (df["Annual_Salary_USD"] < lower_bound)
    | (df["Annual_Salary_USD"] > upper_bound),
    ["Employee_ID", "Department", "Years_Experience", "Annual_Salary_USD"]
].sort_values("Annual_Salary_USD")

salary_outliers

In [ ]:
len(salary_outliers)

Potential outliers should be investigated using context. A high salary may be valid for a senior or specialized employee.

## 17. Box Plot and the Five-Number Summary

In [ ]:
plt.figure(figsize=(9, 4))
plt.boxplot(df["Annual_Salary_USD"], vert=False)
plt.title("Distribution of Annual Salary")
plt.xlabel("Annual Salary (USD)")
plt.tight_layout()
plt.show()

The box represents the middle 50% of salaries. The line inside the box is the median. Points beyond the whiskers are potential outliers under the box-plot rule.

## 18. Skewness

Skewness describes distribution asymmetry:

- near zero: approximately symmetric;
- positive: longer right tail;
- negative: longer left tail.

In [ ]:
salary_skewness = df["Annual_Salary_USD"].skew()
salary_skewness

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df["Annual_Salary_USD"], bins=25, edgecolor="black")
plt.axvline(df["Annual_Salary_USD"].mean(), linestyle="--", label="Mean")
plt.axvline(df["Annual_Salary_USD"].median(), linestyle="-", label="Median")
plt.title("Annual Salary Distribution")
plt.xlabel("Annual Salary (USD)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

A right-skewed distribution commonly has a mean above its median because large observations pull the mean upward.

## 19. Summary with `describe()`

In [ ]:
df[numerical_columns].describe().round(2)

`describe()` provides count, mean, standard deviation, minimum, quartiles, and maximum. It is a useful starting point, but the values still require interpretation.

## 20. Custom Summary Table

In [ ]:
custom_summary = pd.DataFrame({
    "Mean": df[numerical_columns].mean(),
    "Median": df[numerical_columns].median(),
    "Standard_Deviation": df[numerical_columns].std(),
    "Minimum": df[numerical_columns].min(),
    "Q1": df[numerical_columns].quantile(0.25),
    "Q3": df[numerical_columns].quantile(0.75),
    "Maximum": df[numerical_columns].max(),
    "Skewness": df[numerical_columns].skew(),
}).round(2)

custom_summary

## 21. Practical Interpretation

Consider a variable where:

- mean = 72,000;
- median = 61,000;
- maximum = 210,000.

The median may better represent a typical observation because a small number of high values can increase the mean.

## 22. Common Mistakes

1. Assuming the mean is always the best measure.
2. Confusing variance with standard deviation.
3. Treating every extreme value as an error.
4. Interpreting a percentile as a percentage of the variable itself.
5. Reporting statistics without units.
6. Ignoring distribution shape.
7. Comparing standard deviations without considering differences in scale.

## 23. Exercises

1. Calculate the mean, median, and mode of `Age`.
2. Calculate the range and standard deviation of `Performance_Score`.
3. Find the 10th, 25th, 50th, 75th, and 90th percentiles of `Weekly_Hours`.
4. Produce the five-number summary for `Training_Hours`.
5. Use the IQR rule to identify potential outliers in `Absence_Days`.
6. Compare mean salary and median salary within each department.
7. Identify the most variable department using salary standard deviation.
8. Calculate skewness for all numerical variables.
9. Choose one skewed variable and explain whether the mean or median is more representative.
10. Create a histogram and box plot for `Years_Experience`.

In [ ]:
# Write your exercise solutions here.

## 24. Summary

In this notebook, you learned how to:

- calculate measures of central tendency;
- measure variation using range, variance, and standard deviation;
- interpret percentiles and quartiles;
- calculate the IQR and five-number summary;
- identify potential outliers;
- evaluate distribution skewness;
- choose statistics that match the shape and context of the data.

## Next Lesson

The next lesson is **Correlation and Association**, which examines relationships between variables.